# DFL PathCRF + 25fps 경합 재판정

런타임 → 런타임 유형 변경 → GPU를 선택한 뒤 위에서 아래로 실행합니다.

로컬에서 만든 `transition_colab_data.zip` 하나를 선택하면 DFL 7경기와 모델을 준비합니다. 결과는 Google Drive의 `MyDrive/transition_final`에 저장됩니다. 정답과 체크포인트는 로컬 DFL 자료를 사용합니다.

학습 5경기, 검증 J03WQQ, 평가 J03WR9입니다. 경합 후보 기준은 검증에서 정하고 평가 경기의 정답은 마지막 평가 단계에서 사용합니다.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, shutil, zipfile, json, subprocess, sys, hashlib

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/transition_final')
archive = drive_root / 'transition_colab_data.zip'
if not archive.is_file():
    raise FileNotFoundError('Google Drive의 transition_final 폴더에 입력 ZIP을 올리세요.')
print('Drive 입력 파일:', archive, round(archive.stat().st_size / 1024**2, 1), 'MiB')


In [ ]:
project = Path('/content/transition_project')
project.mkdir(exist_ok=True)
if not (project / '.bundle_extracted').exists():
    with zipfile.ZipFile(archive) as z:
        z.extractall(project)
    (project / '.bundle_extracted').touch()
os.chdir(project)
print('준비 완료:', project)


In [ ]:
%pip -q install 'kloppy==3.18.0' pyarrow scipy pandas matplotlib tqdm scikit-learn torch-geometric


## 설정

`EPOCH_WINDOW_CAP=0`은 경합 후보 전체를 사용합니다. 빠른 첫 실험을 원하면 `2000`으로 바꾸세요. 값은 학습 시작 전만 수정하세요. 다른 설정을 실험할 때는 `RUN_NAME`을 바꿔야 기존 체크포인트와 섞이지 않습니다.


In [ ]:
RUN_NAME = 'dfl_5train_1val_1test'
EPOCH_WINDOW_CAP = 0  # 0 = 전체 창, 2000 = 빠른 첫 실험
MAX_EPOCHS = 40
MICROBATCH = 64  # 메모리 부족 시 8로 변경. 총 배치는 64 유지
BENCH_STEPS = 4

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 없습니다. 런타임 유형을 GPU로 변경하세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU 메모리 GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('최대 epoch:', MAX_EPOCHS, 'epoch당 창 상한:', EPOCH_WINDOW_CAP or '전체')


In [ ]:
# 로컬에서 미리 계산한 6경기 PathCRF 결과를 현재 Colab 경로에 연결합니다.
upstream = project / 'pathcrf'
prepared = project / 'data/pathcrf_dfl/transition_head_v1/prepared'
scene = project / 'data/pathcrf_dfl/scene_two_models_v1'
def digest(p):
    with p.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
identity = {
    'pathcrf_commit':'1b7c6f85a6b78c92e4543fde72ca2b86a933f997',
    'pathcrf_checkpoint_sha256':digest(upstream / 'saved/140/model/state_dict_best_acc.pt'),
    'dead_scene_checkpoint_sha256':digest(scene / 'full/best.pt'),
    'dead_scene_decoder_sha256':digest(scene / 'full/fit.json'),
    'candidate_feature_cache':str(prepared.resolve()),
}
key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
destination = project / f'data/transition_final/frozen_{key}'
shutil.copytree(project / 'frozen_for_colab', destination, dirs_exist_ok=True)
run_dir = drive_root / 'runs' / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
configuration = {'RUN_NAME':RUN_NAME,'EPOCH_WINDOW_CAP':EPOCH_WINDOW_CAP,
                 'MAX_EPOCHS':MAX_EPOCHS,'MICROBATCH':MICROBATCH}
config_path = run_dir / 'colab_settings.json'
if config_path.exists() and json.loads(config_path.read_text()) != configuration:
    raise ValueError('기존 설정이 달라요. RUN_NAME을 새 이름으로 바꾸세요.')
config_path.write_text(json.dumps(configuration, ensure_ascii=False, indent=2))
import csv
with (project / 'transition final/parameters.csv').open(newline='', encoding='utf-8') as f:
    parameter_rows = list(csv.DictReader(f))
for row in parameter_rows:
    if row['parameter'] == 'max_epochs':
        row['value'] = str(MAX_EPOCHS)
    if row['parameter'] == 'epoch_window_cap':
        row['value'] = str(EPOCH_WINDOW_CAP)
with (run_dir / 'parameters.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['parameter','value','status','description'])
    writer.writeheader()
    writer.writerows(parameter_rows)
runtime_info = {'gpu':torch.cuda.get_device_name(0), 'torch':str(torch.__version__)}
(run_dir / 'runtime_info.json').write_text(json.dumps(runtime_info, indent=2))
print('파라미터 저장:', run_dir)


In [ ]:
common = [sys.executable, 'transition final/run.py']
options = ['--upstream', str(upstream), '--official-prepared', str(prepared),
           '--device', 'cuda', '--source-device', 'cpu', '--microbatch', str(MICROBATCH),
           '--epochs', str(MAX_EPOCHS), '--epoch-window-cap', str(EPOCH_WINDOW_CAP),
           '--output', str(run_dir)]
def execute(stage, extra=()):
    cmd = common + [stage] + options + list(extra)
    print('실행:', ' '.join(cmd), flush=True)
    with (run_dir / f'{stage}.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f'{stage} 실패. 로그를 확인하세요: {run_dir / (stage + ".log")}')


## 시간 측정

GPU별 배치 시간과 검증 전체 시간을 직접 재고 40 epoch 기준 예상 시간을 표시합니다. 학습 실행 전에 출력된 `benchmark.json`을 확인하세요.


In [ ]:
execute('benchmark', ['--bench-steps', str(BENCH_STEPS), '--full-validation-probe'])
result = json.loads((run_dir / 'benchmark.json').read_text())
print('GPU 모델:', result.get('device_name'))
print('epoch당 창:', result['windows_per_epoch'])
print('epoch당 예상 분:', round(result['projected_epoch_s_min_median_max'][1] / 60, 1))
print('최대 학습 예상 시간:', round(result['projected_training_validation_s_min_median_max'][1] / 3600, 2), '시간')


## 학습

실행하면 epoch마다 검증 Spell F1로 최선의 모델을 저장합니다. 연결이 끊기거나 런타임이 종료되면 같은 설정과 RUN_NAME으로 처음부터 셀을 실행한 뒤 이 셀을 다시 실행하세요. 마지막 완료 epoch의 모델과 Adam 상태에서 이어집니다.


In [ ]:
execute('train')
print('학습 결과:', run_dir)


## 평가 및 CSV 저장

학습을 마치면 테스트 경기에서 예측 D와 정답 D 진단을 각각 평가합니다. 프레임별 예측을 CSV로 변환하고, 결과·설정·학습 이력을 한 번에 확인할 CSV도 저장합니다.


In [ ]:
execute('test')
import pandas as pd
for path in sorted(run_dir.glob('*_J03WR9_H*.parquet')):
    out = path.with_suffix('.csv')
    pd.read_parquet(path).to_csv(out, index=False)
    print('프레임 CSV:', out)
history = json.loads((run_dir / 'training.json').read_text())
pd.json_normalize(history).to_csv(run_dir / 'training_history.csv', index=False)
results = json.loads((run_dir / 'holdout.json').read_text())
rows = []
for mode in ('estimated_D_primary', 'oracle_D_diagnostic'):
    for model in ('base', 'refined'):
        row = {'dead_mask': mode, 'model': model}
        row.update({k:v for k,v in results[mode][model].items() if not isinstance(v, dict)})
        row['candidate_coverage'] = results[mode]['coverage']
        rows.append(row)
pd.DataFrame(rows).to_csv(run_dir / 'holdout_comparison.csv', index=False)
print('비교 CSV:', run_dir / 'holdout_comparison.csv')
print('학습 기록 CSV:', run_dir / 'training_history.csv')
display(pd.DataFrame(rows))


Google Drive에서 `transition_final/runs/RUN_NAME/`을 열면 `parameters.csv`, `colab_settings.json`, `manifest.json`, `training_history.csv`, `holdout_comparison.csv`, 프레임별 CSV, `best.pt`, `last.pt`를 받을 수 있습니다. `last.pt`는 재개에 필요한 Adam 상태를 포함합니다. epoch 도중 중단되면 그 epoch는 다시 실행합니다.
